In [1]:
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient


### połączenie z workspace
ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id="2e6f9eb9-9633-4b76-949e-d54bf09060ba",
    resource_group_name="mai-ml-projects-eus",
    workspace_name="ml-project-covid19"
)

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [3]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

### definicja endpointu
endpoint = ManagedOnlineEndpoint(
    name="covid19-lightgbm-endpoint",
    description="Endpoint dla modelu LightGBM COVID19",
    auth_mode="key"
)

ml_client.begin_create_or_update(endpoint).result()

ManagedOnlineEndpoint({'public_network_access': 'Enabled', 'provisioning_state': 'Succeeded', 'scoring_uri': 'https://covid19-lightgbm-endpoint.eastus.inference.ml.azure.com/score', 'openapi_uri': 'https://covid19-lightgbm-endpoint.eastus.inference.ml.azure.com/swagger.json', 'name': 'covid19-lightgbm-endpoint', 'description': 'Endpoint dla modelu LightGBM COVID19', 'tags': {}, 'properties': {'createdBy': 'Grzegorz Zieliński', 'createdAt': '2025-11-22T19:45:01.695844+0000', 'lastModifiedAt': '2025-11-22T19:45:01.695844+0000', 'azureml.onlineendpointid': '/subscriptions/2e6f9eb9-9633-4b76-949e-d54bf09060ba/resourcegroups/mai-ml-projects-eus/providers/microsoft.machinelearningservices/workspaces/ml-project-covid19/onlineendpoints/covid19-lightgbm-endpoint', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/2e6f9eb9-9633-4b76-949e-d54bf09060ba/providers/Microsoft.MachineLearningServices/locations/eastus/mfeOperationsStatus/oeidp:6f8445aa-fc72-4c43-a457-1f942c455154:2e6

In [31]:
from azure.ai.ml.entities import Environment

env = Environment(
    name="lgbm-env",
    version="1",
    conda_file="./covid19_deploy/lgbm_env.yml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"
)
ml_client.environments.create_or_update(env)

Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'lgbm-env', 'description': None, 'tags': {}, 'properties': {'azureml.labels': 'latest'}, 'print_as_yaml': False, 'id': '/subscriptions/2e6f9eb9-9633-4b76-949e-d54bf09060ba/resourceGroups/mai-ml-projects-eus/providers/Microsoft.MachineLearningServices/workspaces/ml-project-covid19/environments/lgbm-env/versions/1', 'Resource__source_path': '', 'base_path': 'O:\\Projects\\covid19', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x000002AA8040DAD0>, 'serialize': <msrest.serialization.Serializer object at 0x000002AAC61BBE50>, 'version': '1', 'conda_file': {'channels': ['conda-forge'], 'dependencies': ['python=3.9', 'pandas', 'numpy>=1.25.2,<2.0.0', 'scikit-learn', 'lightgbm', 'joblib', 'scipy>=1.11.0', {'pi

In [33]:
### definicja deploymentu (powiązanie z modelem)
from azure.ai.ml.entities import ManagedOnlineDeployment, Environment, CodeConfiguration, BuildContext

deployment = ManagedOnlineDeployment(
    name="covid19-deployment",
    endpoint_name="covid19-lightgbm-endpoint",
    model="lgb_gbdt_best_model:1",  ## nazwa + wersja modelu w workspace
    environment="lgbm-env:1",
    code_configuration=CodeConfiguration(
        code="./covid19_deploy",          ### folder z score.py i lgbm_env.yml
        scoring_script="score.py"  ### plik scoringowy
    ),
    instance_type="Standard_DS2_v2",
    instance_count=1
)

ml_client.begin_create_or_update(deployment).result()

Instance type Standard_DS2_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint covid19-lightgbm-endpoint exists


..................................................................................................................

ManagedOnlineDeployment({'private_network_connection': None, 'package_model': False, 'provisioning_state': 'Succeeded', 'endpoint_name': 'covid19-lightgbm-endpoint', 'type': 'Managed', 'name': 'covid19-deployment', 'description': None, 'tags': {}, 'properties': {'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/2e6f9eb9-9633-4b76-949e-d54bf09060ba/providers/Microsoft.MachineLearningServices/locations/eastus/mfeOperationsStatus/odidp:6f8445aa-fc72-4c43-a457-1f942c455154:a5262cca-6235-4ec5-a871-04557263fb6b?api-version=2023-04-01-preview'}, 'print_as_yaml': False, 'id': '/subscriptions/2e6f9eb9-9633-4b76-949e-d54bf09060ba/resourceGroups/mai-ml-projects-eus/providers/Microsoft.MachineLearningServices/workspaces/ml-project-covid19/onlineEndpoints/covid19-lightgbm-endpoint/deployments/covid19-deployment', 'Resource__source_path': '', 'base_path': 'O:\\Projects\\covid19', 'creation_context': <azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.SystemData object

In [43]:
### testowanie endpointu

## pobranie szczegółów endpointu
endpoint = ml_client.online_endpoints.get(name="covid19-lightgbm-endpoint")
print(endpoint.scoring_uri)

keys = ml_client.online_endpoints.get_keys(name="covid19-lightgbm-endpoint")

## wysyłanie żądania
import requests, json

scoring_uri = endpoint.scoring_uri

headers = {
    "Authorization": f"Bearer {keys.primary_key}",
    "azureml-model-deployment": "covid19-deployment"
}

data = {"data": [[1.2, 3.4, 5.6, 7.8, 1.2, 3.4, 5.6, 7.8, 11.5, 10]]}  ## przykładowe dane wejściowe
response = requests.post(scoring_uri, headers=headers, json=data)

if response.status_code == 200:
    print(response.json())
else:
    print("Error:", response.status_code, response.text)

https://covid19-lightgbm-endpoint.eastus.inference.ml.azure.com/score
{"predictions": [3]}
